**Dataset**
labeled datasset collected from twitter (Lab 1 - Hate Speech.tsv)

**Objective**
classify tweets containing hate speech from other tweets. <br>
0 -> no hate speech <br>
1 -> contains hate speech <br>

**Total Estimated Time = 90-120 Mins**

**Evaluation metric**
macro f1 score

### Import used libraries

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 500)

### Load Dataset

###### Note: search how to load the data from tsv file

In [3]:
df = pd.read_csv("data/Lab_1_Hate_Speech.tsv", sep= "\t")
df.head(100)

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction. #run
1,2,0,@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx. #disapointed #getthanked
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in urð±!!! ðððð ð¦ð¦ð¦
4,5,0,factsguide: society now #motivation
5,6,0,[2/2] huge fan fare and big talking before they leave. chaos and pay disputes when they get there. #allshowandnogo
6,7,0,@user camping tomorrow @user @user @user @user @user @user @user dannyâ¦
7,8,0,the next school year is the year for exams.ð¯ can't think about that ð­ #school #exams #hate #imagine #actorslife #revolutionschool #girl
8,9,0,we won!!! love the land!!! #allin #cavs #champions #cleveland #clevelandcavaliers â¦
9,10,0,@user @user welcome here ! i'm it's so #gr8 !


* looking at data so it includes so many hastags, @, random typings and !!

### Data splitting

It is a good practice to split the data before EDA helps maintain the integrity of the machine learning process, prevents data leakage, simulates real-world scenarios more accurately, and ensures reliable model performance evaluation on unseen data.

In [15]:
print(df["label"].value_counts())
print(f"contains nulls: {df.isnull().sum()}")
print(f"contains duplicates: {df.duplicated().sum()}")

label
0    29322
1     2213
Name: count, dtype: int64
contains nulls: id       0
label    0
tweet    0
dtype: int64
contains duplicates: 0


In [8]:
from sklearn.model_selection import train_test_split

train = int(0.8 * len(df))
test = len(df) - train

train_x, test_x, train_y, test_y = train_test_split(df['tweet'], df['label'], test_size=test, train_size=train, random_state=42, stratify=df['label'])
print(f"train size: {len(train_x)}")
print(f"test size: {len(test_x)}")

train size: 25228
test size: 6307


### EDA on training data

- check NaNs

In [9]:
print(f"null values in train_x: {train_x.isnull().sum()}")
print(f"null values in test_x: {test_x.isnull().sum()}")
print(f"null values in test_y: {test_y.isnull().sum()}")
print(f"null values in train_y: {train_y.isnull().sum()}")

null values in train_x: 0
null values in test_x: 0
null values in test_y: 0
null values in train_y: 0


- check duplicates

In [10]:
print(f"duplicates in train_x: {train_x.duplicated().sum()}")
print(f"duplicates in test_x: {test_x.duplicated().sum()}")
print(f"duplicates in test_y: {test_y.duplicated().sum()}")
print(f"duplicates in train_y: {train_y.duplicated().sum()}")

duplicates in train_x: 1795
duplicates in test_x: 280
duplicates in test_y: 6305
duplicates in train_y: 25226


- show a representative sample of data texts to find out required preprocessing steps

In [13]:
import numpy as np
random_indices = np.random.choice(train_x.index, size=30, replace=False)
for i in random_indices:
	print(f"id: {i}")
	print(f"tweet: {train_x[i]}")
	print(f"label: {train_y[i]}")

id: 26597
tweet: u hu me but i cannot blame u. i gave u the power to hu me. i wish u well tho. #dontbebitter   #joy #love #relationship
label: 0
id: 8814
tweet: whats happening here... pls tag his people.. ó¾´ó¾´  #inshot #girls #cute #summer #blur #sun  â¦...
label: 0
id: 12969
tweet: @user worked on &amp; off over a week &amp; read petercarey's the chemistry of years   #tears #blue #woman not finished yet!
label: 0
id: 16056
tweet: #fotokuapp #funnehanever #goodmorning   langwalangending wheres the nike bag?ð­ð­ð­
label: 0
id: 9643
tweet: #weekend   #workout #fitness #strong #goals its the weekend, don't miss your workout! eatâ¦ 
label: 0
id: 11892
tweet: a list of dark &amp;   #movies with #animals:  | #horror #antagonist #villain #mood
label: 0
id: 18035
tweet: work  #freestyle#hair#hairstyle#hairdresser #thankyou#good#coolâ¦
label: 0
id: 14315
tweet: #payintheusa   polar bear climb racing: angry polar bear climb racing, the polar bear living in cold place 
label: 0
id: 

- check dataset balancing

In [19]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\basil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\basil\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\basil\AppData\Roaming\nltk_data...


True

In [18]:
print(df["label"].value_counts())
print(f"percentage of classes: {df['label'].value_counts(normalize=True)}")

label
0    29322
1     2213
Name: count, dtype: int64
percentage of classes: label
0    0.929824
1    0.070176
Name: proportion, dtype: float64


- Cleaning and Preprocessing are:
    - 1 lower case the data
    - 2 fix contractions "i'll" to be "i will"
    - 3 remove punctuations
    - 4 tokenize text
	- 5 remove stopwords
	- 6 lemmitization

In [22]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
import string
def clean_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer()
    OG_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(OG_tokens)

train_x_cleaned = train_x.apply(clean_text)
print(f"raw_text: {train_x[0]}")
print(f"cleaned_text: {train_x_cleaned[0]}")
    

raw_text: @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run
cleaned_text: user father dysfunctional selfish drag kid dysfunction run


### Cleaning and Preprocessing

#### Extra: use custom scikit-learn Transformers

Using custom transformers in scikit-learn provides flexibility, reusability, and control over the data transformation process, allowing you to seamlessly integrate with scikit-learn's pipelines, enabling you to combine multiple preprocessing steps and modeling into a single workflow. This makes your code more modular, readable, and easier to maintain.

##### link: https://www.andrewvillazon.com/custom-scikit-learn-transformers/

#### Example usage:

In [24]:
from sklearn.base import BaseEstimator, TransformerMixin

class CustomTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, parameter1, parameter2):
        self.parameter1 = parameter1
        self.parameter2 = parameter2
        
        # Add any initialization code here
    
    def fit(self, X, y=None):
        # Add code for fitting the transformer here
        return df
    
    def transform(self, X):
        df = self.fit(X)
        return df
        # Add code for transforming the data here
        transformed_X = X.copy()  # Example: Just copying the data
        
        # Example transformation
        transformed_X['feature1'] = transformed_X['feature1'] * self.parameter1
        transformed_X['feature2'] = transformed_X['feature2'] * self.parameter2
        
        # Do all the needed transformations and data preprocessing here
        
        return transformed_X
    
    def fit_transform(self, X, y=None):
        # This function combines fit and transform
        self.fit(X, y)
        return self.transform(X)

In [ ]:
import re
class clean_preprocess_text(BaseEstimator, TransformerMixin):
	def __init__(self, target_col=None):
		self.target_col = target_col
		self.stop_words = set(stopwords.words("english"))
		self.lemmatizer = WordNetLemmatizer()

	def clean_text(self, text):
		text = str(text).lower()
		text = re.sub(r'[^\x00-\x7F]+', '', text)
		text = contractions.fix(text)
		text = text.translate(str.maketrans("", "", string.punctuation))
		tokens = word_tokenize(text)
		tokens = [word for word in tokens if word not in self.stop_words]
		og_tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
		return " ".join(og_tokens)

	def fit(self, X, y=None):
		return self

	def transform(self, X):
		X_transformed = X.copy()
		if isinstance(X, pd.DataFrame) and self.target_col in X.columns:
			X_transformed[self.target_col] = X_transformed[self.target_col].apply(self.clean_text)
		else:
			X_transformed = X_transformed.apply(self.clean_text) ## the data is series anyway but best practice just incase
		return X_transformed
        

In [31]:
x_transf = clean_preprocess_text(target_col="tweet").fit_transform(train_x)
print(f"raw_text: {train_x[10]}")
print(f"cleaned_text: {x_transf[10]}")

raw_text: â #ireland consumer price index (mom) climbed from previous 0.2% to 0.5% in may   #blog #silver #gold #forex
cleaned_text: â ireland consumer price index mom climbed previous 02 05 may blog silver gold forex


* as i can see from output this awkward text at start need to be removed using regex

In [35]:
x_transf = clean_preprocess_text(target_col="tweet").fit_transform(train_x)
print(f"raw_text: {train_x[10]}")
print(f"cleaned_text: {x_transf[10]}")

raw_text: â #ireland consumer price index (mom) climbed from previous 0.2% to 0.5% in may   #blog #silver #gold #forex
cleaned_text: ireland consumer price index mom climbed previous 02 05 may blog silver gold forex


**You  are doing Great so far!**

### Modelling

#### Extra: use scikit-learn pipline

##### link: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

Using pipelines in scikit-learn promotes better code organization, reproducibility, and efficiency in machine learning workflows.

#### Example usage:

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
CV = CountVectorizer()
TFIDF = TfidfVectorizer()
model = LogisticRegression()

# Create the pipeline
pipeline_cv = Pipeline(steps=[
    ('preprocessing', clean_preprocess_text()),
    ('Vectorizing', CV),
    ('model', model),
])

pipeline_tfidf = Pipeline(steps=[
    ('preprocessing', clean_preprocess_text()),
    ('Vectorizing', TFIDF),
    ('model', model),
])

# Now you can use the pipeline for training and prediction
# pipeline.fit(X_train, y_train)
# pipeline.predict(X_test)

* trying on the countvectorizer with LR

In [ ]:
from sklearn.metrics import accuracy_score
pipeline_cv.fit(train_x, train_y)
pred_cv = pipeline_cv.predict(test_x)
print(pred_cv)
print(accuracy_score(test_y, pred_cv))

[0 0 1 ... 0 0 0]
0.9614713810052323


In [39]:
pipeline_tfidf.fit(train_x, train_y)
pred_tfidf = pipeline_tfidf.predict(test_x)
print(pred_tfidf)
print(accuracy_score(test_y, pred_tfidf))

[0 0 0 ... 0 0 0]
0.9495798319327731


* these accuracy evaluation isnt good because data is imbalanced like we saw earlier but i was testing would it do better than random model that would get 93% ACC or NO

#### Evaluation

**Evaluation metric:**
macro f1 score

Macro F1 score is a useful metric in scenarios where you want to evaluate the overall performance of a multi-class classification model, **particularly when the classes are imbalanced**

![Calculation](https://assets-global.website-files.com/5d7b77b063a9066d83e1209c/639c3d934e82c1195cdf3c60_macro-f1.webp)

In [43]:
from sklearn.metrics import confusion_matrix, classification_report

In [44]:
cm_cv = confusion_matrix(test_y, pred_cv)
cm_tfidf = confusion_matrix(test_y, pred_tfidf)
print("Confusion Matrix for CountVectorizer:")
print(cm_cv)
print("Confusion Matrix for TF-IDF:")
print(cm_tfidf)

Confusion Matrix for CountVectorizer:
[[5840   24]
 [ 219  224]]
Confusion Matrix for TF-IDF:
[[5860    4]
 [ 314  129]]


In [45]:
class_report_cv = classification_report(test_y, pred_cv)
class_report_tfidf = classification_report(test_y, pred_tfidf)
print("Classification Report for CountVectorizer:")
print(class_report_cv)
print("Classification Report for TF-IDF:")
print(class_report_tfidf)

Classification Report for CountVectorizer:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      5864
           1       0.90      0.51      0.65       443

    accuracy                           0.96      6307
   macro avg       0.93      0.75      0.81      6307
weighted avg       0.96      0.96      0.96      6307

Classification Report for TF-IDF:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      5864
           1       0.97      0.29      0.45       443

    accuracy                           0.95      6307
   macro avg       0.96      0.65      0.71      6307
weighted avg       0.95      0.95      0.94      6307



* macro avg f1score for countvectorizer performed better than tfidf 

### Enhancement

- Using different N-grams
- Using different text representation technique
- Hyperparameter tuning

In [54]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "Vectorizing__ngram_range": [(2, 3), (2, 4)],
    "Vectorizing__max_df": [0.85, 0.9],
    "Vectorizing__min_df": [2, 5],
    "Vectorizing__max_features": [None, 5000]
}
grid_search_cv = GridSearchCV(estimator=pipeline_cv, param_grid=param_grid, cv=5)

In [55]:
grid_search_cv.fit(train_x, train_y)
best_params_cv = grid_search_cv.best_params_
print("Best Hyperparameters:", best_params_cv)

Best Hyperparameters: {'Vectorizing__max_df': 0.85, 'Vectorizing__max_features': None, 'Vectorizing__min_df': 2, 'Vectorizing__ngram_range': (2, 4)}


In [56]:
best_estimator_cv = grid_search_cv.best_estimator_
pred_best_estimator_cv = best_estimator_cv.predict(test_x)
best_estimator_report = classification_report(test_y, pred_best_estimator_cv)
print("Classification Report for Best Estimator:")
print(best_estimator_report)

Classification Report for Best Estimator:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      5864
           1       0.99      0.23      0.38       443

    accuracy                           0.95      6307
   macro avg       0.97      0.62      0.67      6307
weighted avg       0.95      0.95      0.93      6307



### Conclusion and final results


In [57]:
print(best_estimator_report)

              precision    recall  f1-score   support

           0       0.95      1.00      0.97      5864
           1       0.99      0.23      0.38       443

    accuracy                           0.95      6307
   macro avg       0.97      0.62      0.67      6307
weighted avg       0.95      0.95      0.93      6307



* the params i tried with gridsearch for CV since it was the better model than TFIDF turned to be bad and performed worse than normal params for CV

#### Done!